# 模块四：可视化报告与交互式 Dashboard (Interactive BI Dashboard)

**本模块核心目标：**
打破传统 Jupyter Notebook 只能输出静态图片 (PNG/JPG) 的局限。
利用 `Plotly` 和自定义的 HTML/CSS 渲染引擎，一键生成带有 KPI 监控卡片、且支持鼠标交互（悬停显示数值、缩放）的独立网页级 BI 看板。
这是向业务方和管理层交付数据分析成果的最高标准形态。

In [1]:
# 1. 开启自动重载
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
import sys

# 引入我们在 src 目录重构好的高级可视化大屏类
sys.path.append('../')
from src.visualizer import DashboardVisualizer

# 2. 加载前三个模块产出的精炼数据 (Processed Data)
print("Loading processed assets for Dashboard generation...")
cohort_matrix = pd.read_parquet("../data/processed/cohort_matrix.parquet")
df_rfm = pd.read_parquet("../data/processed/olist_rfm_segments.parquet")

# 模拟还原模块一的数据质量血统表 (在真实调度引擎中，这部分会存入数据库，此处为了展示看板直接构建)
lineage_data = [
    {"清洗步骤/检查项": "Schema校验: 剔除越界与未知枚举", "淘汰记录数": 12, "剩余记录": 103874},
    {"清洗步骤/检查项": "业务扫描: 剔除状态倒置与时间穿越", "淘汰记录数": 298, "剩余记录": 98874},
    {"清洗步骤/检查项": "对账扫描: 剔除总金额与支付不符订单", "淘汰记录数": 412, "剩余记录": 98462}
]
lineage_df = pd.DataFrame(lineage_data)

print("✅ Data successfully loaded.")

Loading processed assets for Dashboard generation...
✅ Data successfully loaded.


## 步骤一：提取核心业务指标 (KPIs)
高管看看板，第一眼看的一定是最上面的核心数字。我们需要从底层数据中提炼出最有代表性的宏观指标。

In [2]:
# 3. 计算并组装 KPI 字典
total_users = len(df_rfm)
total_revenue = df_rfm['Monetary'].sum()
vip_users = len(df_rfm[df_rfm['Segment'] == 'VIP Loyalists'])

kpi_dict = {
    'total_users': total_users,
    'total_revenue': total_revenue,
    'vip_ratio': f"{(vip_users / total_users) * 100:.1f}%",
    'data_quality_score': "99.2%" # 基于清洗淘汰率计算的综合健康分
}

print("KPIs calculated:")
print(kpi_dict)

KPIs calculated:
{'total_users': 93828, 'total_revenue': np.float64(15600847.780000001), 'vip_ratio': '3.0%', 'data_quality_score': '99.2%'}


## 步骤二：生成交互图表与组装 HTML 网页
调用 `DashboardVisualizer`，生成极具科技感的动态图表，并注入到包含 CSS 布局的 HTML 模板中。

In [3]:
# 4. 生成 Plotly 交互图表的 HTML 代码片段
print("Generating interactive charts...")
plots_dict = {
    'data_quality': DashboardVisualizer.plot_data_quality(lineage_df),
    'cohort_heatmap': DashboardVisualizer.plot_cohort_heatmap(cohort_matrix),
    'rfm_radar': DashboardVisualizer.plot_rfm_radar(df_rfm)
}

# 5. 组装终极 Dashboard 并输出到本地文件
os.makedirs("../outputs", exist_ok=True)
output_file = "../outputs/dashboard.html"

DashboardVisualizer.build_dashboard(
    html_path=output_file,
    kpi_data=kpi_dict,
    plots_html=plots_dict
)

# 在 Notebook 中提供一个点击提示
from IPython.display import display, HTML
display(HTML(f"<h3>🎉 恭喜！Dashboard 已生成！</h3><p>请在您的文件浏览器中找到 <b><a href='{output_file}' target='_blank'>outputs/dashboard.html</a></b> 并用 Chrome 或 Safari 双击打开它。</p>"))

Generating interactive charts...
✅ Dashboard successfully generated at: ../outputs/dashboard.html
